In [ ]:
from singleCAM_IROS._pipeline_support import _handle_dirpaths

from pathlib import Path

from astropy.io import fits
from astropy.io.fits.fitsrec import FITS_rec
from numpy.typing import NDArray
import numpy as np

from bloodmoon.io import simulation_files
from bloodmoon.mask import CodedMaskCamera, codedmask

import darksun as ds

ds.show.set_figures_darkbkg()

In [ ]:
type EnergyRange = tuple[float, float]

def load_fits_data(path: Path, ext: int = 1) -> FITS_rec:
    """Loads the FITS file data from chosen extension."""
    return fits.getdata(path, ext=ext, header=False)

def extract_flx(filepath: str | Path) -> tuple[NDArray, NDArray]:
    """Extracts energy bins and flux arrays from FITS file."""
    flx_rec: FITS_rec = load_fits_data(filepath)
    return flx_rec['EBINS'], flx_rec['FLUX'][:-1]

def get_eband_fract_flx(ebins: NDArray, flux: NDArray, eband: EnergyRange) -> float:
    """Computes the source flux fraction in given energy band."""
    low, high = eband
    band: NDArray = np.where((ebins >= low) & (ebins <= high))[0][:-1]
    return flux[band].sum() / flux.sum()

EBINS, MFLXARR = extract_flx('/mnt/dbb8f47e-da06-47bf-8ef5-038092af70f7/Edos_Magnificent_Manor/PhD_AASS/Coding/IROS_Data/Simulations/camera_settings/RXTE-ASM_BeppoSAX-WFC_catalog_2-50keV_MeanAvgFluxCrablike.fits')

In [ ]:
MASK_FITS: str = "wfm_mask_NTHT_20250725.fits"

SKYFIELD: str = "IROSDummy"
DATA_FITS: str = "iros_benchmark_2-50keV_mask_050_1040x17_1ks"

ID_CAMERA_A: str = "cam1a"
ID_CAMERA_B: str = "cam1b"
DATASET: str = "reconstructed"

UPS_X: int = 5
UPS_Y: int = 1

VIGNETTING: bool = True
PSFY: bool = True

In [ ]:
# load filepaths
mask_path, simul_data, save_path = _handle_dirpaths(
    mask=MASK_FITS,
    skyfield=SKYFIELD,
    simul=DATA_FITS,
)
wfm: CodedMaskCamera = codedmask(mask_path, UPS_X, UPS_Y)
filepaths: dict[str, dict[str, Path]] = simulation_files(simul_data)

# data from camera A
sdlA = ds.get_data(filepaths[ID_CAMERA_A][DATASET])
catalogueA = ds.get_catalogue(filepaths[ID_CAMERA_A]['sources'])

# data from camera B
sdlB = ds.get_data(filepaths[ID_CAMERA_B][DATASET])
catalogueB = ds.get_catalogue(filepaths[ID_CAMERA_B]['sources'])

In [ ]:
from typing import Callable
from dataclasses import dataclass

from scipy.optimize import curve_fit

from bloodmoon.coords import pos2shift
from bloodmoon.optim import process_skyimg
from bloodmoon.optim import model_sky


type CoordSky = tuple[float, float]

@dataclass
class OptResults:
    """Source optimisation result container."""
    counts: float
    coords: dict[EnergyRange, CoordSky]


def _ModelShiftFluence(
    camera: CodedMaskCamera,
    pos: tuple[int, int],
    ebands: tuple[EnergyRange, ...],
    vignetting: bool = True,
    psfy: bool = True,
) -> Callable[[NDArray, *tuple[float, ...]], NDArray]:
    """
    Initialises the source model.
    """
    FRACTS: tuple[float, ...] = tuple(
        get_eband_fract_flx(EBINS, MFLXARR, eband) for eband in ebands
    )

    def project(sx: float, sy: float, cts: float) -> NDArray:
        """Generates a source sky image from coords."""
        return model_sky(camera, sx, sy, cts, vignetting, psfy)

    def f(x: NDArray, sx0, sy0, sx1, sy1, sx2, sy2, cts) -> NDArray:
        """Models the source sky image."""
        modeled: NDArray = (
            project(sx0, sy0, FRACTS[0]),
            project(sx1, sy1, FRACTS[1]),
            project(sx2, sy2, FRACTS[2]),
        ) * cts
        return process_skyimg(camera, modeled, pos)

def optimize(
    camera: CodedMaskCamera,
    sky: NDArray,
    arg_sky: tuple[int, int],
    model: Callable[[NDArray, *tuple[float, ...]], NDArray],
    verbose: bool = False,
) -> tuple[float, float, float]:
    """
    Performs the optimization to fit a point source model to sky image data.
    """
    px_dim_x, px_dim_y = (
        camera.specs.mask_deltax / camera.upscale_f.x,
        camera.specs.mask_deltay / camera.upscale_f.y,
    )
    camera_coding_power = 0.85

    model_shift_flux = _ModelShiftFluence(camera, arg_sky, vignetting, psfy)
    sx_start, sy_start = pos2shift(camera, *arg_sky)
    sky_peak = sky[*arg_sky]
    fluence_start = (
        sky_peak / camera_coding_power if psfy else sky_peak
    )
    sky_ydata = process_skyimg(camera, sky, arg_sky)
    
    # - the shifts are allowed to fluctuate in a 3 x 3 pixel box since
    #   the extracted position is close enough to the true source pos
    #   Also, since multiple sources may be superimposed or close, the
    #   optimisation procedure may introduce biases in the source fit
    # - the fluence cannot be smaller than the one observed at the peak,
    #   and we insert a lower value just for precaution (if simulating
    #   for example an infinite detector spatial resolution)
    results, _ = curve_fit(
        model_shift_flux,
        xdata=np.arange(len(sky_ydata)),
        ydata=sky_ydata,
        p0=[sx_start, sy_start, fluence_start],
        bounds=[
            (
                max(sx_start - 1.5 * px_dim_x, camera.bins_sky.x[0]),
                max(sy_start - 1.5 * px_dim_y, camera.bins_sky.y[0]),
                0.95 * sky_peak,
            ),
            (
                min(sx_start + 1.5 * px_dim_x, camera.bins_sky.x[-1]),
                min(sy_start + 1.5 * px_dim_y, camera.bins_sky.y[-1]),
                1.25 * sky_peak,
            ),
        ],
    )
    # store the final optimized positions and fluence
    sx, sy, fluence = map(float, results)

    if verbose:
        print(
            f'\n'
            f'## Optimisation Results:\n'
            f'  - fluence START: {fluence_start}\n'
            f'  - shifts START (x, y): {sx_start}, {sy_start}\n'

            f'  - fluence OPTIM.: {fluence}\n'
            f'  - shifts OPTIM. (x, y): {sx}, {sy}\n'

            f'  - fluence GAIN %: {(fluence - fluence_start) * 100 / fluence_start:.3f}\n'
            f'  - shift_x GAIN %: {np.sign(sx_start) * (sx - sx_start) * 100 / sx_start:.3f}\n'
            f'  - shift_y GAIN %: {np.sign(sy_start) * (sy - sy_start) * 100 / sy_start:.3f}\n'
        )

    return sx, sy, fluence